In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [3]:
try:
    eng = engs.get_engine()
    text_qry = text(engs.load_query('qry_olos.sql'))
    df = pd.read_sql(text_qry, eng)
except Exception as e:
    print (f'Erro ao executar consulta: {e}')

In [4]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [5]:
df.columns

Index(['area', 'base_type', 'campaign_id', 'tablename', 'disposition_nivel_1',
       'data', 'hour', 'tentativas', 'atendidas', 'day_week', 'semana_mes'],
      dtype='object')

In [14]:
df_today = df.copy()
df_today = df_today[df_today['data'] == '2026-05-07']
df_today = df_today.groupby(['tablename']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
df_today['tx'] = ((df_today['atendidas'] /  df_today['tentativas']) * 100 ).round(2)
df_today.sort_values(by='tx',ascending=False)


,tablename,tentativas,atendidas,tx
0,,6,5,83.33
6,_1605_FISIOTERAPIA_BASELEADS_ENGAJADOSBLACKFRI...,335,15,4.48
22,_1605_PSICOLOGIA_PAGEVIEWENGAJADOMARKETING_060...,311,13,4.18
7,_1605_FISIOTERAPIA_BASELEADS_ENGAJADOSBLACKFRI...,263,9,3.42
14,_1605_PSICOLOGIA_BASELEADS_05052026_V500_pt2,478,16,3.35
10,_1605_MEDICINA_ARTMED_PAGEVIEW_07052026_V500_PT2,399,13,3.26
3,_1605_ENFERMAGEM_MATERIALRICO_EBOOOK_PROTOCOLO...,401,13,3.24
12,_1605_MULTI_EVENTO_MANGUITOROTADOR_06052026_V694,535,17,3.18
1,_1605_ENFERMAGEM_EVENTO_ATUALIZACAOSAUDEMATERN...,811,25,3.08
15,_1605_PSICOLOGIA_EVENTO_IIICONGRESSOONLINEPSIC...,337,10,2.97


In [13]:
df_tab = df.copy()
df_tab = df_tab[df_tab['data'] == '2026-05-07']
df_tab = df_tab.groupby(['disposition_nivel_1']).agg(
    quantidade = ('disposition_nivel_1','count')
).reset_index()
df_tab.sort_values(by='quantidade',ascending=False)


,disposition_nivel_1,quantidade
18,NoAnswer,101
5,CarrierMessage,100
1,Announcement,99
25,Silence,92
3,BadNumber,90
11,IMPRODUTIVO LIGACAO CAIU,64
2,BadLine,57
26,TransferModule,47
4,Busy,44
22,RETORNO AGENDADO,26
